In [1]:
import pandas as pd
import requests
import json
import time
import datetime

In [5]:
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/111.0.0.0 Safari/537.36'}

In [2]:
df_source = pd.read_json("best_cities_france.json")
df_source = df_source.rename(columns={0: "city"})
df_source.head()

,city
0,Mont Saint Michel
1,St Malo
2,Bayeux
3,Le Havre
4,Rouen


In [ ]:
df = df_source.copy()
for index, row in df.iterrows():
    print(index, row["city"])
    res = requests.get(f"https://nominatim.openstreetmap.org/search?q={row['city']},France&format=json", headers=headers)
    city = res.json()[0]
    df.loc[index, "lat"] = city["lat"]
    df.loc[index, "lon"] = city["lon"]
    time.sleep(1)
df.head()
df.to_csv("cities_with_geoposition.csv", index=False)

0 Mont Saint Michel
1 St Malo
2 Bayeux
3 Le Havre
4 Rouen
5 Paris
6 Amiens
7 Lille
8 Strasbourg
9 Chateau du Haut Koenigsbourg
10 Colmar
11 Eguisheim
12 Besancon
13 Dijon
14 Annecy
15 Grenoble
16 Lyon
17 Gorges du Verdon
18 Bormes les Mimosas
19 Cassis
20 Marseille
21 Aix en Provence
22 Avignon
23 Uzes
24 Nimes
25 Aigues Mortes
26 Saintes Maries de la mer
27 Collioure
28 Carcassonne
29 Ariege
30 Toulouse
31 Montauban
32 Biarritz
33 Bayonne
34 La Rochelle


In [3]:
df = pd.read_csv("cities_with_geoposition.csv")
df.head()

,city,lat,lon
0,Mont Saint Michel,48.635954,-1.511460
1,St Malo,48.649518,-2.026041
2,Bayeux,49.276462,-0.702474
3,Le Havre,49.493898,0.107973
4,Rouen,49.440459,1.093966


In [ ]:
list_weather_data = []

for index, row in df.iterrows():
    res_weather = requests.get(f"https://api.openweathermap.org/data/2.5/forecast?lat={row['lat']}&lon={row['lon']}&units=metric&exclude=current,minutely,hourly,alerts&appid=4656ed7337e689999007412af6a4dafe", headers=headers)
    res_weather_json = res_weather.json()
    
    for res in res_weather_json['list']:
        weather_entry = {
            "city": row['city'],
            "date": datetime.datetime.fromtimestamp(res['dt']).strftime('%d/%m/%Y'),
            "hour": datetime.datetime.fromtimestamp(res['dt']).strftime('%H:%M'),
            "temp": res['main']['temp'],
            "perc_humidity": res['main']['humidity'],
            "prob_rain": res['pop'],
            "volume_rain": res.get('rain', {}).get('3h', 0),
            "wind_speed": res['wind']['speed'],
            "perc_cloud": res['clouds']['all']
        }
        list_weather_data.append(weather_entry)
    time.sleep(1)

df_weather = pd.DataFrame(list_weather_data)
print(df_weather.head())
df_weather.to_csv("weather_forecast.csv", index=False)

                city        date   hour  temp  perc_humidity  prob_rain  \
0  Mont Saint Michel  09/02/2026  16:00  8.79             93       1.00   
1  Mont Saint Michel  09/02/2026  19:00  9.40             94       1.00   
2  Mont Saint Michel  09/02/2026  22:00  9.44             95       0.41   
3  Mont Saint Michel  10/02/2026  01:00  9.56             95       1.00   
4  Mont Saint Michel  10/02/2026  04:00  9.71             91       0.55   

   volume_rain  wind_speed  perc_cloud  
0         4.41        9.97         100  
1         1.92        6.10         100  
2         0.31        4.61          91  
3         2.24        5.57          93  
4         0.37        6.50         100  


In [ ]:
df_weather = pd.read_csv("weather_forecast.csv")
df_weather.head()

In [9]:
df_weather_groupby = df_weather.groupby(['city', 'date']).agg({'temp': ['mean', 'min', 'max'], 'perc_humidity': 'mean', 'prob_rain': 'max', 'volume_rain': {'mean', 'max'}, 'wind_speed': 'max', 'perc_cloud': 'mean'}).reset_index()
df_weather_groupby.head()

city        date       temp               perc_humidity prob_rain  \
                                   mean    min    max          mean       max   
0  Aigues Mortes  09/02/2026  12.093333  11.18  13.02        74.000       0.0   
1  Aigues Mortes  10/02/2026  10.595000   9.53  11.56        90.250       1.0   
2  Aigues Mortes  11/02/2026  13.522500  12.18  14.97        78.125       1.0   
3  Aigues Mortes  12/02/2026  12.112500  10.86  12.88        73.625       1.0   
4  Aigues Mortes  13/02/2026  10.371250   9.49  11.85        75.375       1.0   

  volume_rain       wind_speed perc_cloud  
         mean   max        max       mean  
0     0.00000  0.00       1.67  32.333333  
1     0.83875  3.46       7.30  86.750000  
2     0.11125  0.32       8.93  89.875000  
3     0.46750  3.16      14.09  66.500000  
4     0.20750  0.84       7.01  90.875000

In [ ]:
# Scale temp & volume_rain
# temp max 26 (based on health preocupations)
# volume_rain is ok when below 2.5–4 mm per hour
df_weather_groupby['score'] = df_weather_groupby['temp']['max'] * 0.5 + df_weather_groupby['perc_humidity']['mean'] * 0.2 + df_weather_groupby['prob_rain']['max'] * 0.2 + df_weather_groupby['volume_rain']['max'] * 0.8 + df_weather_groupby['wind_speed']['max'] * 0.6 + df_weather_groupby['perc_cloud']['mean'] * 0.1
df_weather_groupby.head()

city        date       temp               perc_humidity prob_rain  \
                                   mean    min    max          mean       max   
0  Aigues Mortes  09/02/2026  12.093333  11.18  13.02        74.000       0.0   
1  Aigues Mortes  10/02/2026  10.595000   9.53  11.56        90.250       1.0   
2  Aigues Mortes  11/02/2026  13.522500  12.18  14.97        78.125       1.0   
3  Aigues Mortes  12/02/2026  12.112500  10.86  12.88        73.625       1.0   
4  Aigues Mortes  13/02/2026  10.371250   9.49  11.85        75.375       1.0   

  volume_rain       wind_speed perc_cloud      score  
         mean   max        max       mean             
0     0.00000  0.00       1.67  32.333333  25.545333  
1     0.83875  3.46       7.30  86.750000  39.853000  
2     0.11125  0.32       8.93  89.875000  37.911500  
3     0.46750  3.16      14.09  66.500000  38.997000  
4     0.20750  0.84       7.01  90.875000  35.165500